# 01 — Data Preprocessing

**Phase:** Data Loading → Audit → Cleaning → Feature Engineering

**What this notebook does:**
1. Loads the raw Excel file
2. Audits shape, types, missing values, duplicates
3. Cleans the data (drop IDs, standardize text, handle missing, cap outliers)
4. Engineers new features (`route`, `duration_bucket`)
5. **Saves a clean CSV** for the next notebooks

**Run this first.** Then open `02_eda_and_statistics.ipynb`.


In [18]:
import warnings
warnings.filterwarnings('ignore')

import random
import numpy as np
import pandas as pd
import re

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print(f'Seed set to {SEED}')

Seed set to 42


## 1. Load Raw Data

Change the path below to match your machine.

In [19]:
# UPDATE THIS PATH to your actual file location
DATA_PATH = r'D:\MS_Data_Science_Arden_university\Machine learning\ML_portfolio\ML_Project_report\Data\Flight_dataset[4039].xlsx'
SHEET_NAME = 'in'

df_raw = pd.read_excel(DATA_PATH, sheet_name=SHEET_NAME)
print('Loaded shape:', df_raw.shape)
print('Columns:', list(df_raw.columns))
df_raw.head()

Loaded shape: (300153, 12)
Columns: ['Unnamed: 0', 'airline', 'flight', 'source_city', 'departure_time', 'stops', 'arrival_time', 'destination_city', 'class', 'duration', 'days_left', 'price']


,Unnamed: 0,airline,flight,source_city,departure_time,stops,arrival_time,destination_city,class,duration,days_left,price
0,0,SpiceJet,SG-8709,Delhi,Evening,zero,Night,Mumbai,Economy,2.17,1.0,5953.0
1,1,SpiceJet,SG-8157,Delhi,Early_Morning,zero,Morning,Mumbai,Economy,2.33,1.0,5953.0
2,2,AirAsia,I5-764,Delhi,Early_Morning,zero,Early_Morning,Mumbai,Economy,NaN,1.0,5956.0
3,3,Vistara,UK-995,Delhi,Morning,zero,Afternoon,Mumbai,Economy,2.25,1.0,5955.0
4,4,Vistara,UK-963,Delhi,Morning,zero,Morning,Mumbai,Economy,2.33,1.0,5955.0


## 2. Data Audit

Check missing values, data types, and duplicates **before** touching anything.

In [20]:
# Missing values
missing = df_raw.isna().sum().rename('missing_count').to_frame()
missing['missing_pct'] = (missing['missing_count'] / len(df_raw) * 100).round(4)
print('Missing values:')
display(missing[missing['missing_count'] > 0])

# Data types
print('Data types:')
display(df_raw.dtypes.rename('dtype').to_frame())

# Duplicates
print(f'Duplicate rows: {df_raw.duplicated().sum()}')

Missing values:


,missing_count,missing_pct
duration,12,0.0040
days_left,5,0.0017
price,5,0.0017


Data types:


,dtype
Unnamed: 0,int64
airline,object
flight,object
source_city,object
departure_time,object
stops,object
arrival_time,object
destination_city,object
class,object
duration,float64


Duplicate rows: 0


## 3. Preprocessing Pipeline

**Order matters:**
1. Drop useless columns (`Unnamed: 0`, `flight` ID)
2. Standardize text (lowercase, underscores)
3. Remove exact duplicate rows
4. Convert duration to minutes
5. Handle missing values (median for numbers, mode for categories)
6. Cap outliers using IQR (keep rows, limit extreme values)

In [21]:
# Create BEFORE snapshot
df_pre = df_raw.copy()
pre_snapshot = {
    'rows': len(df_pre),
    'columns': df_pre.shape[1],
    'missing_values': int(df_pre.isna().sum().sum()),
    'duplicate_rows': int(df_pre.duplicated().sum())
}
print('=== BEFORE PREPROCESSING ===')
print(f'Rows: {pre_snapshot["rows"]}, Columns: {pre_snapshot["columns"]}')
print(f'Missing values: {pre_snapshot["missing_values"]}, Duplicates: {pre_snapshot["duplicate_rows"]}')

=== BEFORE PREPROCESSING ===
Rows: 300153, Columns: 12
Missing values: 22, Duplicates: 0


In [22]:
# 3.1 Drop useless columns
df = df_raw.copy()
removed_cols = []

if 'Unnamed: 0' in df.columns:
    df = df.drop(columns=['Unnamed: 0'])
    removed_cols.append('Unnamed: 0')

if 'flight' in df.columns:
    df = df.drop(columns=['flight'])
    removed_cols.append('flight')

print(f'Removed columns: {removed_cols}')

Removed columns: ['Unnamed: 0', 'flight']


In [23]:
# 3.2 Standardize text categories
categorical_cols = df.select_dtypes(include=['object', 'string']).columns.tolist()

for col in categorical_cols:
    df[col] = (
        df[col]
        .astype('string')
        .str.strip()
        .str.lower()
        .str.replace(' ', '_', regex=False)
    )

print(f'Standardised: {categorical_cols}')
df.head(3)

Standardised: ['airline', 'source_city', 'departure_time', 'stops', 'arrival_time', 'destination_city', 'class']


,airline,source_city,departure_time,stops,arrival_time,destination_city,class,duration,days_left,price
0,spicejet,delhi,evening,zero,night,mumbai,economy,2.17,1.0,5953.0
1,spicejet,delhi,early_morning,zero,morning,mumbai,economy,2.33,1.0,5953.0
2,airasia,delhi,early_morning,zero,early_morning,mumbai,economy,NaN,1.0,5956.0


In [24]:
# 3.3 Remove exact duplicates
rows_before = len(df)
df = df.drop_duplicates().copy()
duplicates_removed = rows_before - len(df)
print(f'Rows before: {rows_before}')
print(f'Duplicates removed: {duplicates_removed}')
print(f'Rows after: {len(df)}')

Rows before: 300153
Duplicates removed: 2212
Rows after: 297941


In [25]:
# 3.4 Convert duration to minutes
def duration_to_minutes(value):
    if pd.isna(value):
        return np.nan
    if isinstance(value, (int, float, np.number)):
        return float(value) * 60
    text = str(value).strip().lower()
    if re.fullmatch(r'\d+(\.\d+)?', text):
        return float(text) * 60
    hours, minutes = 0, 0
    h_match = re.search(r'(\d+)\s*h', text)
    m_match = re.search(r'(\d+)\s*m', text)
    if h_match:
        hours = int(h_match.group(1))
    if m_match:
        minutes = int(m_match.group(1))
    return (hours * 60) + minutes if (h_match or m_match) else np.nan

df['duration_minutes'] = df['duration'].apply(duration_to_minutes)
df['duration_hours'] = df['duration_minutes'] / 60
df = df.drop(columns=['duration'])

print('Duration converted to minutes. Sample:')
df[['duration_minutes', 'duration_hours']].head()

Duration converted to minutes. Sample:


,duration_minutes,duration_hours
0,130.2,2.17
1,139.8,2.33
2,NaN,NaN
3,135.0,2.25
4,139.8,2.33


In [26]:
# 3.5 Handle missing values
# Drop rows where target (price) is missing
target_missing = int(df['price'].isna().sum())
df = df.dropna(subset=['price']).copy()
print(f'Rows dropped (missing price): {target_missing}')

# Impute numerical features with median
for col in ['duration_minutes', 'days_left']:
    if df[col].isna().any():
        median_val = df[col].median()
        df[col] = df[col].fillna(median_val)
        print(f'{col}: imputed missing with median {median_val:.2f}')

# Impute categorical features with mode
for col in categorical_cols:
    if col in df.columns and df[col].isna().any():
        mode_val = df[col].mode(dropna=True)
        if not mode_val.empty:
            df[col] = df[col].fillna(mode_val.iloc[0])
            print(f'{col}: imputed missing with mode {mode_val.iloc[0]}')

print(f'Final shape: {df.shape}')
print(f'Remaining missing values: {df.isna().sum().sum()}')

Rows dropped (missing price): 5
duration_minutes: imputed missing with median 679.80
days_left: imputed missing with median 26.00
Final shape: (297936, 11)
Remaining missing values: 12


In [27]:
# 3.6 Outlier capping (IQR method)
for feature in ['price', 'duration_minutes']:
    q1 = df[feature].quantile(0.25)
    q3 = df[feature].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    outlier_mask = (df[feature] < lower) | (df[feature] > upper)
    df[f'{feature}_outlier_iqr'] = outlier_mask
    df[f'{feature}_capped'] = df[feature].clip(lower=lower, upper=upper)

    print(f'{feature}: {outlier_mask.sum()} outliers capped between {lower:.1f} and {upper:.1f}')

# Use capped price as target going forward
df['price'] = df['price_capped']
print('Outlier capping complete.')

price: 123 outliers capped between -51801.5 and 99114.5
duration_minutes: 2226 outliers capped between -417.3 and 1802.7
Outlier capping complete.


In [28]:
# 4. Feature Engineering
df['route'] = df['source_city'] + '_to_' + df['destination_city']
df['duration_bucket'] = pd.cut(
    df['duration_minutes'],
    bins=[-np.inf, 180, 420, 660, np.inf],
    labels=['short_<=3h', 'medium_3to7h', 'long_7to11h', 'very_long_>11h']
).astype('string')

print(f'Unique routes: {df["route"].nunique()}')
print(f'Duration buckets: {df["duration_bucket"].value_counts().to_dict()}')
df[['route', 'duration_minutes', 'duration_bucket']].head()

Unique routes: 30
Duration buckets: {'very_long_>11h': 153770, 'long_7to11h': 67050, 'medium_3to7h': 42970, 'short_<=3h': 34146}


,route,duration_minutes,duration_bucket
0,delhi_to_mumbai,130.2,short_<=3h
1,delhi_to_mumbai,139.8,short_<=3h
2,delhi_to_mumbai,679.8,very_long_>11h
3,delhi_to_mumbai,135.0,short_<=3h
4,delhi_to_mumbai,139.8,short_<=3h


In [29]:
# 5. Save clean data for next notebooks
# Update this path to your project folder
OUTPUT_PATH = r'D:\MS_Data_Science_Arden_university\Machine learning\ML_portfolio\ML_Project_report\data\flight_data_cleaned.csv'

# Create data folder if it doesn't exist
import os
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)

df.to_csv(OUTPUT_PATH, index=False)
print(f'Clean data saved to: {OUTPUT_PATH}')
print(f'Final dataset: {df.shape[0]} rows × {df.shape[1]} columns')
print('✅ Notebook 01 complete. Now open 02_eda_and_statistics.ipynb')

Clean data saved to: D:\MS_Data_Science_Arden_university\Machine learning\ML_portfolio\ML_Project_report\data\flight_data_cleaned.csv
Final dataset: 297936 rows × 17 columns
✅ Notebook 01 complete. Now open 02_eda_and_statistics.ipynb
